# 🧩 Séance 5 — Structure 3D des protéines (AlphaFold 2) et impact des mutations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nevermind78/IA_Biotech/blob/main/notebooks/S5_alphafold_mutations.ipynb)

**Module : Intelligence Artificielle et Biotechnologie — Atelier 5 / 5 (2h) + mini-projet**

> ⚙️ **Avant de commencer :** `Exécution` → `Modifier le type d'exécution` → **GPU T4**, puis exécutez
> **immédiatement** les cellules de la section 0 : l'installation de ColabFold tourne en arrière-plan
> pendant que vous travaillez sur la partie 1.

## 🧬 Le problème biologique

La **fonction** d'une protéine dépend de sa **forme 3D**. Une seule mutation ponctuelle peut
déstabiliser ce repliement (maladies génétiques, perte d'activité d'une enzyme industrielle…) ou au
contraire l'améliorer (ingénierie des protéines). Déterminer une structure expérimentalement
(cristallographie, cryo-EM) prend des mois ; **AlphaFold 2** (DeepMind, prix Nobel de chimie 2024) la
prédit en quelques minutes.

> **Question :** que prédit AlphaFold pour une protéine **mutée** ? Peut-on s'en servir pour évaluer
> l'impact fonctionnel d'une mutation ? Quelles sont les **limites** de ces modèles ?

## 🎯 Objectifs
1. Visualiser une structure en 3D avec **py3Dmol** et lire le score de confiance **pLDDT**
2. Prédire une structure avec **ColabFold** (AlphaFold 2 + MSA rapide MMseqs2)
3. Comparer prédiction et structure **expérimentale** (RMSD)
4. Introduire des **mutations**, comparer sauvage vs mutant, croiser avec un **modèle de langage** (ESM-2)
5. **Mini-projet (45 min)** en binôme : présenter l'impact d'une mutation et les limites du modèle

## 🧭 Légende
▶️ exécuter · ✍️ à vous · 🔎 interpréter · 💡 solution (cellule masquée)

⏱️ **Déroulé indicatif :** installation + visualisation AlphaFold DB (20 min) · prédiction ColabFold (25 min) · mutations & ESM-2 (30 min) · **mini-projet (45 min)**

---
## 0. Installation ▶️ (à lancer tout de suite)

In [ ]:
# ✍️ Moteur de prédiction :
#   • "ColabFold (AlphaFold2)" — le vrai AlphaFold 2 avec alignement multiple (recommandé, GPU)
#   • "ESMFold local (GPU)"    — plan B : modèle de langage, tourne sur le GPU de Colab, sans MSA ni serveur
#   • "ESMFold API (sans GPU)" — plan C : service public gratuit, mais souvent saturé
MOTEUR = "ColabFold (AlphaFold2)"  #@param ["ColabFold (AlphaFold2)", "ESMFold local (GPU)", "ESMFold API (sans GPU)"]

%pip install -q py3Dmol biopython

In [ ]:
# ▶️ Imports (Biopython est importé AVANT l'installation de ColabFold)
import glob, io, json, os, re, subprocess, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import torch
import py3Dmol
from Bio.PDB import PDBParser, PDBIO, Superimposer

sns.set_theme(style="whitegrid")
EN_COLAB = "google.colab" in sys.modules

def lancer_installation_colabfold():
    """Installe ColabFold en arrière-plan (commande officielle du notebook ColabFold)."""
    if os.path.isfile("COLABFOLD_READY"):
        print("✅ ColabFold déjà installé"); return
    script = ("pip install -q --no-warn-conflicts "
              "'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold' "
              "> install_colabfold.log 2>&1 "
              "&& ln -sf /usr/local/lib/python3.*/dist-packages/colabfold colabfold "
              "&& ln -sf /usr/local/lib/python3.*/dist-packages/alphafold alphafold "
              "&& touch COLABFOLD_READY")
    subprocess.Popen(["bash", "-c", script])
    print("⏳ Installation de ColabFold lancée en arrière-plan (3 à 5 min). Continuez la partie 1 !")

if MOTEUR.startswith("ColabFold"):
    if EN_COLAB:
        lancer_installation_colabfold()
    else:
        print("ℹ️ ColabFold est prévu pour Google Colab (Linux + GPU). Hors Colab, choisissez ESMFold.")
print("✅ Imports OK — moteur choisi :", MOTEUR)

---
## 1. Lire une structure 3D et son score de confiance ▶️

L'**AlphaFold Protein Structure Database** contient déjà plus de 200 millions de structures prédites.
Récupérons celle de la **chaîne β de l'hémoglobine humaine** (UniProt `P68871`).

Dans un fichier PDB produit par AlphaFold, la colonne « B-factor » contient le **pLDDT** (0-100) :

| pLDDT | Interprétation | Couleur |
|---|---|---|
| > 90 | très haute confiance (position des chaînes latérales fiable) | 🟦 bleu foncé |
| 70 – 90 | confiance (squelette fiable) | 🟦 bleu clair |
| 50 – 70 | faible confiance | 🟨 jaune |
| < 50 | très faible : souvent une **région désordonnée** | 🟧 orange |

In [ ]:
# ▶️ Fonctions utilitaires fournies
def telecharger(url, chemin):
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with open(chemin, "wb") as f:
        f.write(r.content)
    return chemin

def structure_alphafold_db(uniprot):
    info = requests.get(f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot}", timeout=60).json()[0]
    return telecharger(info["pdbUrl"], f"AF-{uniprot}.pdb"), info

def plddt_par_residu(chemin_pdb):
    """pLDDT (colonne B-factor) de chaque résidu, lu sur le carbone alpha."""
    s = PDBParser(QUIET=True).get_structure("p", chemin_pdb)
    return pd.Series({r.id[1]: r["CA"].get_bfactor() for r in s.get_residues() if "CA" in r})

def couleur_plddt(v):
    return "#0053D6" if v > 90 else "#65CBF3" if v > 70 else "#FFDB13" if v > 50 else "#FF7D45"

def vue_3d(chemin_pdb, couleur="plddt", residus_en_batonnets=(), largeur=650, hauteur=450):
    vue = py3Dmol.view(width=largeur, height=hauteur)
    vue.addModel(open(chemin_pdb).read(), "pdb")
    if couleur == "plddt":
        for resi, v in plddt_par_residu(chemin_pdb).items():
            vue.setStyle({"resi": int(resi)}, {"cartoon": {"color": couleur_plddt(v)}})
    else:
        vue.setStyle({"cartoon": {"color": "spectrum"}})
    for resi in residus_en_batonnets:
        vue.addStyle({"resi": int(resi)}, {"stick": {"colorscheme": "magentaCarbon"}})
        vue.addLabel(str(resi), {"fontSize": 11, "backgroundOpacity": 0.6}, {"resi": int(resi), "atom": "CA"})
    vue.zoomTo()
    return vue

pdb_hbb, info_hbb = structure_alphafold_db("P68871")
print("Fichier :", pdb_hbb, "— pLDDT moyen :", info_hbb.get("globalMetricValue"))
vue_3d(pdb_hbb, residus_en_batonnets=[7]).show()

In [ ]:
# ▶️ Profil de confiance le long de la séquence
def tracer_plddt(series_dict, titre="pLDDT par résidu"):
    plt.figure(figsize=(11, 3.5))
    for bande, c in [((90, 100), "#0053D6"), ((70, 90), "#65CBF3"), ((50, 70), "#FFDB13"), ((0, 50), "#FF7D45")]:
        plt.axhspan(*bande, color=c, alpha=0.12)
    for nom, s in series_dict.items():
        plt.plot(s.index, s.to_numpy(), lw=2, label=nom)
    plt.ylim(0, 100); plt.xlabel("Position du résidu"); plt.ylabel("pLDDT"); plt.title(titre)
    plt.legend(); plt.show()

tracer_plddt({"Hémoglobine β (AlphaFold DB)": plddt_par_residu(pdb_hbb)})

### 🔎 Question 1
Faites tourner la molécule à la souris (molette = zoom). Quelle **structure secondaire** domine
(hélices α ou feuillets β) ? Le résidu 7 (en magenta) est le **glutamate** muté dans la
**drépanocytose** : est-il enfoui au cœur de la protéine ou exposé en **surface** ?

*Votre réponse :* …

---
## 2. Prédire une structure avec ColabFold ▶️

Notre protéine modèle : le **domaine B1 de la protéine G** de streptocoque (**GB1**, 56 acides aminés),
une petite protéine très étudiée en ingénierie des protéines. Sa structure a été résolue par
cristallographie (PDB **1PGA**) : on pourra donc **vérifier** la prédiction.

ColabFold enchaîne : (1) recherche de séquences homologues → **alignement multiple (MSA)** via le serveur
MMseqs2 ; (2) **AlphaFold 2** prédit la structure ; (3) écriture des fichiers PDB et des scores.

In [ ]:
# ▶️ Vérifier que l'installation est terminée (attend au maximum 8 minutes)
def attendre_colabfold(max_minutes=8):
    if not MOTEUR.startswith("ColabFold"):
        print(f"Moteur « {MOTEUR} » : pas d'installation ColabFold nécessaire ✅"); return True
    debut = time.time()
    while not os.path.isfile("COLABFOLD_READY"):
        if time.time() - debut > max_minutes * 60:
            print("❌ Installation trop longue. Consultez install_colabfold.log ou passez MOTEUR sur ESMFold.")
            return False
        print(f"\r⏳ installation en cours… {int(time.time() - debut)} s", end="")
        time.sleep(10)
    print("\n✅ ColabFold est prêt"); return True

attendre_colabfold()

In [ ]:
# ▶️ Plan B : ESMFold exécuté localement (sur le GPU de Colab), sans serveur externe
_tok_fold = _mod_fold = None
DEVICE_FOLD = "cuda" if __import__("torch").cuda.is_available() else "cpu"

def modele_esmfold():
    """Charge ESMFold une seule fois (≈ 2,6 Go téléchargés au premier appel)."""
    global _tok_fold, _mod_fold
    if _mod_fold is not None:
        return _mod_fold
    import torch
    from transformers import AutoTokenizer, EsmForProteinFolding
    print("⏳ Chargement d'ESMFold (≈ 2,6 Go au premier appel)…")
    _tok_fold = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
    _mod_fold = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
    _mod_fold = _mod_fold.to(DEVICE_FOLD).eval()
    if DEVICE_FOLD == "cuda":
        _mod_fold.esm = _mod_fold.esm.half()      # moitié de la mémoire, même résultat
        _mod_fold.trunk.set_chunk_size(64)        # tient dans les 15 Go d'un GPU T4
    print("✅ ESMFold prêt")
    return _mod_fold

In [ ]:
# ▶️ Fonction de prédiction (ColabFold, ESMFold local ou ESMFold API selon MOTEUR)
def predire_structure(sequence, nom, n_modeles=1, n_recyclages=3, mode_msa="mmseqs2_uniref_env"):
    """Renvoie un dictionnaire : chemin du PDB, pLDDT par résidu, PAE (si ColabFold), pTM."""
    sequence = sequence.strip().upper()
    os.makedirs("predictions", exist_ok=True)
    t0 = time.time()
    if MOTEUR.startswith("ColabFold"):
        fasta = f"predictions/{nom}.fasta"
        with open(fasta, "w") as f:
            f.write(f">{nom}\n{sequence}\n")
        dossier = f"predictions/{nom}"
        cmd = ["colabfold_batch", "--num-models", str(n_modeles), "--num-recycle", str(n_recyclages),
               "--msa-mode", mode_msa, fasta, dossier]
        print("$", " ".join(cmd))
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for ligne in proc.stdout:                       # affiche la progression en direct
            if any(m in ligne for m in ["Query", "MSA", "rank_001", "pLDDT", "Done", "rror"]):
                print("   ", ligne.rstrip()[-120:])
        proc.wait()
        pdb = sorted(glob.glob(f"{dossier}/{nom}_unrelaxed_rank_001*.pdb"))[0]
        scores = json.load(open(sorted(glob.glob(f"{dossier}/{nom}_scores_rank_001*.json"))[0]))
        pae, ptm = np.array(scores["pae"]), scores.get("ptm")
    elif MOTEUR.startswith("ESMFold local"):
        modele_esmfold()                                     # chargé une seule fois
        ids = _tok_fold([sequence], return_tensors="pt", add_special_tokens=False)["input_ids"]
        with torch.no_grad():
            sortie = _mod_fold(ids.to(DEVICE_FOLD))
        pdb = f"predictions/{nom}_esmfold.pdb"
        with open(pdb, "w") as f:
            f.write(_mod_fold.output_to_pdb(sortie)[0])
        pae, ptm = None, None
    else:
        # L'API publique d'ESMFold est parfois saturée : on réessaie plusieurs fois
        r = None
        for essai in range(1, 5):
            try:
                r = requests.post("https://api.esmatlas.com/foldSequence/v1/pdb/", data=sequence, timeout=300)
                r.raise_for_status()
                break
            except Exception as e:
                print(f"   essai {essai}/4 : {type(e).__name__} — nouvelle tentative dans 20 s…")
                r = None
                time.sleep(20)
        if r is None:
            raise RuntimeError("ESMFold ne répond pas. Réessayez dans quelques minutes, "
                               "ou utilisez le notebook officiel ColabFold (voir la page de la séance).")
        lignes = []
        for l in r.text.splitlines():                   # ESMFold renvoie un pLDDT entre 0 et 1 → 0-100
            if l.startswith("ATOM") and float(l[60:66]) <= 1.0:
                l = l[:60] + f"{float(l[60:66]) * 100:6.2f}" + l[66:]
            lignes.append(l)
        pdb = f"predictions/{nom}_esmfold.pdb"
        with open(pdb, "w") as f:
            f.write("\n".join(lignes) + "\n")
        pae, ptm = None, None
    plddt = plddt_par_residu(pdb)
    print(f"✅ {nom} : pLDDT moyen = {plddt.mean():.1f}  ({time.time() - t0:.0f} s)")
    return {"nom": nom, "sequence": sequence, "pdb": pdb, "plddt": plddt, "pae": pae, "ptm": ptm}

GB1 = "MTYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE"
print("Longueur de GB1 :", len(GB1))

In [ ]:
# ▶️ Prédiction de la protéine sauvage (1re fois : ~3-5 min avec le téléchargement des poids AlphaFold)
sauvage = predire_structure(GB1, "gb1_sauvage")
vue_3d(sauvage["pdb"]).show()

In [ ]:
# ▶️ Matrice PAE (Predicted Aligned Error) : erreur attendue sur la position relative de 2 résidus
if sauvage["pae"] is not None:
    plt.figure(figsize=(5, 4.2))
    plt.imshow(sauvage["pae"], cmap="Greens_r", vmin=0, vmax=30)
    plt.colorbar(label="erreur attendue (Å)")
    plt.title(f"PAE — GB1 sauvage (pTM = {sauvage['ptm']:.2f})"); plt.xlabel("résidu"); plt.ylabel("résidu")
    plt.show()
else:
    print("ESMFold (API) ne fournit pas de PAE.")

### Comparaison avec la structure expérimentale ▶️
Le **RMSD** (*root-mean-square deviation*) mesure l'écart moyen, en ångströms (1 Å = 0,1 nm), entre les
atomes de deux structures **après superposition optimale**. RMSD < 1-2 Å : structures quasi identiques.

In [ ]:
def superposer(pdb_reference, pdb_mobile, sortie=None):
    """Superpose les carbones alpha (même numérotation) et renvoie RMSD global + écart par résidu."""
    p = PDBParser(QUIET=True)
    ref = p.get_structure("ref", pdb_reference)
    mob = p.get_structure("mob", pdb_mobile)
    ca_ref = {r.id[1]: r["CA"] for r in next(ref.get_models()).get_residues() if "CA" in r and r.id[0] == " "}
    ca_mob = {r.id[1]: r["CA"] for r in next(mob.get_models()).get_residues() if "CA" in r and r.id[0] == " "}
    communs = sorted(set(ca_ref) & set(ca_mob))
    sup = Superimposer()
    sup.set_atoms([ca_ref[i] for i in communs], [ca_mob[i] for i in communs])
    sup.apply(list(mob.get_atoms()))
    ecarts = pd.Series({i: ca_ref[i] - ca_mob[i] for i in communs})   # distance en Å après superposition
    if sortie:
        io_pdb = PDBIO(); io_pdb.set_structure(mob); io_pdb.save(sortie)
    return sup.rms, ecarts

telecharger("https://files.rcsb.org/download/1PGA.pdb", "1PGA.pdb")
rmsd_exp, ecarts_exp = superposer("1PGA.pdb", sauvage["pdb"], sortie="gb1_sauvage_aligne.pdb")
print(f"RMSD prédiction vs cristallographie (1PGA) : {rmsd_exp:.2f} Å sur {len(ecarts_exp)} carbones alpha")

vue = py3Dmol.view(width=650, height=450)
vue.addModel(open("1PGA.pdb").read(), "pdb")
vue.addModel(open("gb1_sauvage_aligne.pdb").read(), "pdb")
vue.setStyle({"model": 0}, {"cartoon": {"color": "lightgrey"}})
vue.setStyle({"model": 1}, {"cartoon": {"color": "#0053D6", "opacity": 0.8}})
vue.zoomTo(); vue.show()
print("Gris : cristallographie (1PGA) — Bleu : prédiction")

### 🔎 Question 2
Le RMSD est-il faible ? Dans quelles régions la prédiction s'écarte-t-elle le plus de l'expérience ?
Ces régions ont-elles un pLDDT plus bas ? (Le pLDDT est une **auto-évaluation** du modèle : utile,
mais ce n'est **pas une mesure expérimentale**.)

*Votre réponse :* …

### ✍️ Exercice 1 — À quoi sert l'alignement multiple (MSA) ?
Relancez la prédiction de GB1 **sans MSA** (`mode_msa="single_sequence"`, ColabFold uniquement) et
comparez le pLDDT moyen et le RMSD. Qu'en déduisez-vous sur la source de l'« intelligence »
d'AlphaFold ?

In [ ]:
#@title 💡 Solution de l'exercice 1
if MOTEUR.startswith("ColabFold"):
    sans_msa = predire_structure(GB1, "gb1_sans_msa", mode_msa="single_sequence")
    rmsd_sans, _ = superposer("1PGA.pdb", sans_msa["pdb"])
    print(f"Avec MSA : pLDDT {sauvage['plddt'].mean():.1f}, RMSD {rmsd_exp:.2f} Å")
    print(f"Sans MSA : pLDDT {sans_msa['plddt'].mean():.1f}, RMSD {rmsd_sans:.2f} Å")
    tracer_plddt({"avec MSA": sauvage["plddt"], "sans MSA": sans_msa["plddt"]}, "Effet de l'alignement multiple")
else:
    print("ESMFold n'utilise pas de MSA : c'est un modèle de langage (comme ESM-2, séance 3) qui la remplace !")

---
## 3. Introduire des mutations ✍️

Notation : **`W43A`** = le tryptophane (W) en position 43 est remplacé par une alanine (A).
La fonction `muter` **vérifie** que l'acide aminé d'origine est bien le bon (erreur fréquente !).

Quelques positions de GB1 :
- **cœur hydrophobe** (enfoui) : `Y3`, `L5`, `L7`, `A26`, `F30`, `A34`, `W43`, `Y45`, `F52`, `V54`
- **surface** : `K10`, `T16`, `E19`, `D22`, `K28`, `Q32`, `D36`, `D40`, `E42`
- acides aminés « cassants » : la **proline** (P) brise les hélices et les feuillets, la **glycine** (G)
  est très flexible, un résidu **chargé** (D, E, K, R) dans le cœur hydrophobe est très déstabilisant.

In [ ]:
def muter(sequence, mutations):
    """mutations : liste du type ["W43A", "F30P"] (numérotation à partir de 1)."""
    seq = list(sequence)
    for m in mutations:
        aa_origine, position, aa_nouveau = m[0], int(m[1:-1]), m[-1]
        if seq[position - 1] != aa_origine:
            raise ValueError(f"{m} : la position {position} contient {seq[position - 1]}, pas {aa_origine}")
        seq[position - 1] = aa_nouveau
    return "".join(seq)

# ✍️ Choisissez vos mutations
MUTATIONS = "W43P, F30P, Y45D"  #@param {type:"string"}
liste_mutations = [m.strip() for m in MUTATIONS.split(",") if m.strip()]
positions_mutees = [int(m[1:-1]) for m in liste_mutations]

seq_mutant = muter(GB1, liste_mutations)
print("Sauvage :", GB1)
print("Mutant  :", "".join(b if a == b else b.lower() for a, b in zip(GB1, seq_mutant)), "(minuscules = mutations)")

In [ ]:
# ▶️ Prédiction du mutant et comparaison
nom_mutant = "gb1_" + "_".join(liste_mutations)
mutant = predire_structure(seq_mutant, nom_mutant)
rmsd_mut, ecarts_mut = superposer(sauvage["pdb"], mutant["pdb"], sortie=f"{nom_mutant}_aligne.pdb")

comparaison = pd.DataFrame({
    "pLDDT moyen": [sauvage["plddt"].mean(), mutant["plddt"].mean()],
    "pLDDT min": [sauvage["plddt"].min(), mutant["plddt"].min()],
    "pTM": [sauvage["ptm"], mutant["ptm"]],
    "RMSD vs sauvage (Å)": [0.0, rmsd_mut],
}, index=["sauvage", "mutant " + ", ".join(liste_mutations)]).round(2)
display(comparaison)

tracer_plddt({"sauvage": sauvage["plddt"], "mutant": mutant["plddt"]}, "Confiance : sauvage vs mutant")
plt.figure(figsize=(11, 2.8))
plt.bar(ecarts_mut.index, ecarts_mut.to_numpy(), color="tab:red")
for p in positions_mutees:
    plt.axvline(p, ls="--", c="k", lw=0.8)
plt.ylabel("écart (Å)"); plt.xlabel("Position"); plt.title("Déplacement de chaque résidu (mutant vs sauvage)")
plt.show()

In [ ]:
# ▶️ Superposition 3D : gris = sauvage, couleur pLDDT = mutant, magenta = résidus mutés
vue = py3Dmol.view(width=700, height=480)
vue.addModel(open(sauvage["pdb"]).read(), "pdb")
vue.addModel(open(f"{nom_mutant}_aligne.pdb").read(), "pdb")
vue.setStyle({"model": 0}, {"cartoon": {"color": "lightgrey", "opacity": 0.7}})
for resi, v in mutant["plddt"].items():
    vue.setStyle({"model": 1, "resi": int(resi)}, {"cartoon": {"color": couleur_plddt(v)}})
for p in positions_mutees:
    vue.addStyle({"model": 1, "resi": p}, {"stick": {"colorscheme": "magentaCarbon"}})
vue.zoomTo(); vue.show()

### 🔎 Question 3
La structure du mutant est-elle **très différente** de la protéine sauvage ? Le pLDDT a-t-il baissé
autour des mutations ? ⚠️ AlphaFold a été entraîné à prédire la structure **des protéines naturelles
repliées** : il a tendance à prédire **le même repliement** même pour un mutant qui, en réalité, ne
se replierait plus. **Une structure prédite n'est pas une preuve de stabilité !**

*Votre réponse :* …

---
## 4. Un deuxième avis : le modèle de langage ESM-2 ▶️

À la séance 3, ESM-2 a appris à **deviner un acide aminé masqué**. On peut lui demander : *« à cette
position, quelle est la probabilité de l'acide aminé muté par rapport à l'acide aminé sauvage ? »*

$$\text{score}(\text{mutation}) = \log P(\text{aa muté}) - \log P(\text{aa sauvage})$$

Score **très négatif** = mutation « jamais vue par l'évolution » → probablement **délétère**.
Ce score *zéro-shot* (sans aucun entraînement supplémentaire) est corrélé aux mesures expérimentales
d'effets de mutations (Meier *et al.*, 2021).

In [ ]:
%pip install -q transformers
import torch
from transformers import AutoTokenizer, EsmForMaskedLM

MODELE_ESM = "facebook/esm2_t30_150M_UR50D"  #@param ["facebook/esm2_t12_35M_UR50D", "facebook/esm2_t30_150M_UR50D", "facebook/esm2_t33_650M_UR50D"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok_esm = AutoTokenizer.from_pretrained(MODELE_ESM)
mlm_esm = EsmForMaskedLM.from_pretrained(MODELE_ESM).to(DEVICE).eval()
AA = list("ACDEFGHIKLMNPQRSTVWY")

@torch.no_grad()
def carte_mutations(sequence, lot=16):
    """Score ESM-2 de toutes les mutations ponctuelles possibles (20 lignes × L colonnes)."""
    ids = tok_esm(sequence, return_tensors="pt")["input_ids"].to(DEVICE)
    L, ids_aa = len(sequence), tok_esm.convert_tokens_to_ids(AA)
    logp = torch.zeros(L, 20)
    for debut in range(0, L, lot):
        positions = list(range(debut, min(debut + lot, L)))
        masque = ids.repeat(len(positions), 1)
        for k, i in enumerate(positions):
            masque[k, i + 1] = tok_esm.mask_token_id           # +1 : le jeton <cls> est en position 0
        logits = mlm_esm(masque).logits
        for k, i in enumerate(positions):
            logp[i] = torch.log_softmax(logits[k, i + 1], dim=-1)[ids_aa].cpu()
    sauvage_idx = torch.tensor([AA.index(a) for a in sequence])
    score = logp - logp[torch.arange(L), sauvage_idx].unsqueeze(1)
    return pd.DataFrame(score.numpy().T, index=AA, columns=[f"{a}{i + 1}" for i, a in enumerate(sequence)])

carte_gb1 = carte_mutations(GB1)
plt.figure(figsize=(16, 5))
sns.heatmap(carte_gb1, cmap="RdBu", center=0, vmin=-10, vmax=4, cbar_kws={"label": "score ESM-2"})
plt.title("Paysage mutationnel de GB1 prédit par ESM-2 (bleu = toléré, rouge = délétère)")
plt.xlabel("Position (acide aminé sauvage)"); plt.ylabel("Acide aminé muté")
plt.show()

def score_esm(carte, mutations):
    return {m: round(float(carte.loc[m[-1], m[:-1]]), 2) for m in mutations}

print("Score ESM-2 de vos mutations :", score_esm(carte_gb1, liste_mutations))
print("Sensibilité moyenne par position (les 8 plus sensibles) :")
print(carte_gb1.mean().sort_values().head(8).round(2).to_dict())

### 🔎 Question 4
Les positions les plus « sensibles » selon ESM-2 sont-elles celles du **cœur hydrophobe** ?
Les deux modèles (AlphaFold : structure ; ESM-2 : plausibilité évolutive) sont-ils d'accord sur
vos mutations ?

*Votre réponse :* …

---
## 5. 🏁 Mini-projet en binôme (45 min)

**Consigne.** Choisissez **une protéine** et **1 à 3 mutations**, prédisez les structures sauvage et
mutante, croisez avec ESM-2 (et AlphaMissense pour une protéine humaine), puis présentez en
**3 minutes** (3 diapositives max) :
1. **La question** : protéine, rôle biologique, mutation(s) choisie(s) et pourquoi ;
2. **Les résultats** : pLDDT sauvage vs mutant, RMSD, une capture 3D, score ESM-2 ;
3. **L'interprétation et les limites** : que peut-on conclure ? que **ne peut-on pas** conclure ?
   (biais des données d'entraînement, boîte noire, pas de dynamique ni d'environnement cellulaire,
   **validation expérimentale indispensable** : stabilité thermique, activité, cristallographie…)

| Sujet | Séquence | Idées de mutations |
|---|---|---|
| **A. GB1** (protéine modèle) | `GB1` ci-dessus | cœur `W43`, `F52`, `Y3` vs surface `K10`, `E19` |
| **B. Hémoglobine β** (drépanocytose) | UniProt `P68871` | `E7V` (notée **E6V** en clinique car la Met initiale est retirée) |
| **C. Ubiquitine** (signalisation) | UniProt `P0CG48`, résidus 1-76 | `I44A` (surface d'interaction), `L67S` (cœur) |
| **D. Votre protéine** (< 150 aa) | UniProt de votre choix | justifiez votre choix ! |

In [ ]:
# ✍️ Sujet du binôme : collez la séquence et les mutations
NOM_PROJET = "hbb"  #@param {type:"string"}
UNIPROT = "P68871"  #@param {type:"string"}
MUTATIONS_PROJET = "E7V"  #@param {type:"string"}
DEBUT, FIN = 1, None   # ✍️ pour ne garder qu'un domaine, ex. DEBUT, FIN = 1, 76 pour l'ubiquitine

fasta_txt = requests.get(f"https://rest.uniprot.org/uniprotkb/{UNIPROT}.fasta", timeout=60).text
seq_projet = "".join(fasta_txt.splitlines()[1:])[DEBUT - 1:FIN]
mut_projet = [m.strip() for m in MUTATIONS_PROJET.split(",") if m.strip()]
print(fasta_txt.splitlines()[0])
print(f"Longueur : {len(seq_projet)} aa — mutations : {mut_projet}")

In [ ]:
# ▶️ Pipeline du mini-projet : sauvage + mutant + comparaison + ESM-2 (+ AlphaMissense si disponible)
proj_sauvage = predire_structure(seq_projet, f"{NOM_PROJET}_sauvage")
proj_mutant = predire_structure(muter(seq_projet, mut_projet), f"{NOM_PROJET}_{'_'.join(mut_projet)}")
rmsd_proj, ecarts_proj = superposer(proj_sauvage["pdb"], proj_mutant["pdb"], sortie=f"{NOM_PROJET}_mutant_aligne.pdb")
carte_proj = carte_mutations(seq_projet)

# AlphaMissense (DeepMind, 2023) : pathogénicité prédite de chaque variant humain (fourni par AlphaFold DB)
try:
    info = requests.get(f"https://alphafold.ebi.ac.uk/api/prediction/{UNIPROT}", timeout=60).json()[0]
    am_csv = requests.get(info["amAnnotationsUrl"], timeout=60).text
    am = pd.read_csv(io.StringIO(am_csv)).set_index("protein_variant")
    # AlphaMissense utilise la numérotation UniProt complète : on décale si on a coupé la séquence
    am_scores = {m: am.loc[f"{m[0]}{int(m[1:-1]) + DEBUT - 1}{m[-1]}", "am_pathogenicity"] for m in mut_projet}
except Exception:
    am_scores = {m: None for m in mut_projet}

bilan = pd.DataFrame({
    "pLDDT moyen sauvage": proj_sauvage["plddt"].mean(),
    "pLDDT moyen mutant": proj_mutant["plddt"].mean(),
    "RMSD mutant vs sauvage (Å)": rmsd_proj,
    "score ESM-2": [score_esm(carte_proj, [m])[m] for m in mut_projet],
    "AlphaMissense (0 bénin → 1 pathogène)": [am_scores[m] for m in mut_projet],
}, index=mut_projet).round(2)
display(bilan)
tracer_plddt({"sauvage": proj_sauvage["plddt"], "mutant": proj_mutant["plddt"]}, f"{NOM_PROJET} : sauvage vs mutant")
vue_3d(f"{NOM_PROJET}_mutant_aligne.pdb", residus_en_batonnets=[int(m[1:-1]) for m in mut_projet]).show()

### 🔎 Pour la présentation — le cas de la drépanocytose (sujet B)
La mutation **E6V** de l'hémoglobine β est l'une des mutations pathogènes les plus célèbres.
Regardez ce que prédisent AlphaFold, ESM-2 et AlphaMissense. Sont-ils capables de « voir » la maladie ?

*Indice :* la drépanocytose n'est pas due à un mauvais repliement : la valine hydrophobe exposée en
surface fait **polymériser** l'hémoglobine désoxygénée en longues fibres qui déforment les globules
rouges. Un modèle qui ne regarde qu'**une seule chaîne** et **un seul état** peut-il le prédire ?

---
## 📝 Bilan du module

| Séance | Donnée | Modèle | Message clé |
|---|---|---|---|
| 1 | Tableau de descripteurs | Random Forest | évaluer correctement, classes déséquilibrées |
| 2 | Matrice d'expression | ACP, UMAP, K-Means | découvrir des groupes sans étiquettes |
| 3 | Séquences | k-mers, ESM-2 | les modèles pré-entraînés apportent de la connaissance |
| 4 | Images | Cellpose (CNN), Isolation Forest | automatiser et quantifier |
| 5 | Structures | AlphaFold 2, ESM-2 | puissance **et limites** : la validation expérimentale reste indispensable |

**Livrable à rendre :** ce notebook exécuté + les 3 diapositives du binôme.